In [25]:
import boto3

from config import REGION

ec2=boto3.client("ec2",region_name=REGION)

def get_instances():
    response=ec2.describe_instances()
    instances=[]
    for reservation in response["Reservations"]:
        for instance in reservation["Instances"]:
            name="NO NAME"
            if "Tags" in instance:
                for tag in instance["Tags"]:
                    if tag["Key"]=="Name":
                        name=tag["Value"]
            instances.append({
                "Name":name,
                "Instances ID":instance["InstanceId"],
                "State":instance["InstanceType"],
                "Instance Type":instance["InstanceType"],
                "Public IP":instance.get("PublicIpAddress","-"),
                "Private IP":instance.get("PrivateIpAddress","-"),
                "Availability Zone":instance["Placement"]["AvailabilityZone"],
                "Launch Time":instance["LaunchTime"]
            })        
def get_instance(instance_id):
    response=ec2.describe_instances(
        InstanceIds=[Instance_id],
        
    )
    return response["Reservations"][0]["Instances"][0]



In [26]:
response=ec2.describe_instances()

In [27]:
print(response)

{'Reservations': [{'ReservationId': 'r-04adb1a00ea6f805e', 'OwnerId': '440779547439', 'Groups': [], 'Instances': [{'Architecture': 'x86_64', 'BlockDeviceMappings': [{'DeviceName': '/dev/xvda', 'Ebs': {'AttachTime': datetime.datetime(2026, 9, 6, 12, 2, 28, tzinfo=tzutc()), 'DeleteOnTermination': True, 'Status': 'attached', 'VolumeId': 'vol-0d34a3c07ad19238a', 'EbsCardIndex': 0}}], 'ClientToken': 'edbd0da5-9278-4061-991b-cf2407231e34', 'EbsOptimized': True, 'EnaSupport': True, 'Hypervisor': 'xen', 'NetworkInterfaces': [{'Attachment': {'AttachTime': datetime.datetime(2026, 9, 6, 12, 2, 27, tzinfo=tzutc()), 'AttachmentId': 'eni-attach-09a2517f1be64577e', 'DeleteOnTermination': True, 'DeviceIndex': 0, 'Status': 'attached', 'NetworkCardIndex': 0}, 'Description': '', 'Groups': [{'GroupId': 'sg-07d752bcc7b18ceb0', 'GroupName': 'launch-wizard-1'}], 'Ipv6Addresses': [], 'MacAddress': '0a:99:e8:c8:2b:61', 'NetworkInterfaceId': 'eni-06f976b5db54b2eaa', 'OwnerId': '440779547439', 'PrivateDnsName': 

In [28]:
#for cloud watch

import boto3

import pandas as pd

from datetime import datetime, timedelta

from config import REGION

cloudwatch=boto3.client("cloudwatch",region_name=REGION)

def get_cpu_metrics(instance_id):
    end=datetime.utcnow()
    start=end-timedelta(hours=1)

    response=cloudwatch.get_metrics_statistics(
        Namespace="AWS/EC2",
        MetriName="CPUUtilization",
        Dimensions=
            [
                {
                    "Name":"InstacneId",
                    "Value":instance_id,
                }
            ],
        StartTime=start,
        EndTime=end,
        Period=300,
        Statistics=["Average"]
    )

    points=sorted(response["Datapoints"],key=lambda x:x ["TimeStamp"])

    if len(points)==0:
        return 0,pd.DataFrame()
    df=pd.DataFrame(points)
    df["Time"]=df["TimeStamp"]
    df["CPU"]=df("Average")

    latest=round(df.iloc[-1]["CPU"],2)

    return latest,df

In [29]:
import boto3

from config import REGION

from config import SNS_TOPIC_ARN

sns=boto3.client("sns",region_name=REGION)

def send_alert(instanceid,cpu):
    sns.publish(
        TopicArn=SNS_TOPIC_ARN,
        subject="AWS ec2 alert",
        message=f"""
        EC2 alert

        Instance;{instance_id}

        CPU_Usage:{cpu}%

        Time: High cpu utilisation

        """
        )

In [30]:
def health(cpu):
    if cpu<70:
        return "Healthy"
    return "warning"    

In [61]:
import streamlit as st

import pandas as pd

import plotly.express as px

from aws.ec2 import get_instances

from aws.ec2 import get_instance

from aws.cloudwatch import get_cpu_metrics

from aws.sns import send_alert

from utlis.helper import health

st.set_page_config(page_title="AWS ec2 monitor",layout="wide")

st.title("AWS ec2 health monitoring dashboard")

instances=get_instances()
df=pd.DataFrame(instances)

st.subheader("AWS ec2 instances")

st.dataframe(
    df,
    use_container_width=True
)

instance = st.selectbox(
    "Select ec2 instance",
    df["Instance ID"]
)

details=get_instance(instance)

cpu,cpu_df=get_cpu_metrics(instance)

col1,col2,col3=st.columns(3)

col1.metrics(
    "cpu usage",
    f"{cpu}%"
)

col2.metrics(
    "State",
    details["State"]["Name"]
)

col3.metrics(
    "type",
    details["InstanceType"]
)

st.divider()

left,right=st.columns(2)

with left:
    st.write("EC2 details")
    st.write("Instance ID",details["InstanceId"])
    st.write(
        "Public IP:  ",details.get("PublicIpAddress","-")
    )
    st.write(
        "Private IP :",details.get("PrivateIpAddress","-")
    )
with right:
    st.write("AWS right")
    st.write("Availability zone:",details["Placement"]["AvailabilityZone"])
    st.write("LAUNCH TIME",details["LaunchTime"])
    
st.subheader("CPU utilisation")

if not cpu_df.empty:
    fig=px.line(
        cpu_df,
        x="TIME",
        y="CPU",
        marker=True,
        title="Last 1hr CPU usage"
    )

    st.plotly_chart(
        fig,
        use_container_width=True,
    )
else:
    st.warning("NO cloud Watch me")
status=health(cpu)

if status=="Healthy":
    st.success(status)
else:
    st.error(status)

col1,col2=st.columns(2)

with col1:
    if st.button("Refresh"):
        st.rerun()
with col2:
    if st.button("Send Alert"):
        send_alert(instance,cpu)
        st.success("Alert sent")
    
    

2026-11-23 17:52:10.513 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 17:52:10.517 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 17:52:10.518 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 17:52:10.520 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 17:52:11.084 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 17:52:11.085 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 17:52:11.087 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-11-23 17:52:11.087 Please replace `use_container_width` with `width`.

`use_container_width` will be removed afte

KeyError: 'Instance ID'